# Partie 3 — Spark SQL et DataFrames

**Objectif :** refaire et enrichir l'analyse de la Partie 2 avec l'API structurée (DataFrame + SQL).

Prérequis : `orders_history.jsonl` et `restaurants.json` sont dans `lake/bronze/` (voir `demarrer.ps1`).

> La Spark UI (DAG, onglet **SQL / DataFrame**) est sur http://localhost:4040 tant que la session tourne,
> puis dans le **History Server** : http://localhost:18080.

In [1]:
import json, os, time
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               IntegerType, TimestampType)
from spark_session import get_spark, LAKE

spark = get_spark("FastFood-Partie3-SQL")
sc = spark.sparkContext

HISTORY = f"{LAKE}/bronze/history/orders_history.jsonl"
RESTAURANTS = f"{LAKE}/bronze/reference/restaurants.json"
SILVER = f"{LAKE}/silver/orders_history"
RESULTS = "/home/jovyan/work/results"
os.makedirs(RESULTS, exist_ok=True)

timings = {}
def chrono(label, action):
    t0 = time.perf_counter()
    res = action()
    timings[label] = round(time.perf_counter() - t0, 3)
    print(f"[temps] {label} : {timings[label]} s")
    return res

## 1. Lecture avec un schéma explicite (`StructType`), sans inférence

Sans schéma, Spark lirait tout le fichier une première fois pour deviner les types (job supplémentaire)
et pourrait se tromper (ex. `event_time` lu comme simple chaîne). Avec un schéma explicite, la lecture se fait
en un seul passage et les types sont garantis. Ce même schéma servira en Partie 4 (`from_json`).

In [2]:
ORDER_SCHEMA = StructType([
    StructField("order_id",      StringType(),    False),
    StructField("restaurant_id", StringType(),    True),
    StructField("customer_id",   StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("amount_mad",    DoubleType(),    True),
    StructField("items_count",   IntegerType(),   True),
    StructField("distance_km",   DoubleType(),    True),
    StructField("courier_id",    StringType(),    True),
    StructField("status",        StringType(),    True),
    StructField("event_time",    TimestampType(), True),
])

RESTAURANT_SCHEMA = StructType([
    StructField("restaurant_id", StringType(), False),
    StructField("name",          StringType(), True),
    StructField("city",          StringType(), True),
    StructField("category",      StringType(), True),
    StructField("latitude",      DoubleType(), True),
    StructField("longitude",     DoubleType(), True),
])

orders = (spark.read.schema(ORDER_SCHEMA)
          .option("mode", "DROPMALFORMED")                        # lignes invalides ignorées
          .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ssX")
          .json(HISTORY))
restaurants = spark.read.schema(RESTAURANT_SCHEMA).json(RESTAURANTS)

orders.printSchema()
print("événements :", orders.count(), "| restaurants :", restaurants.count())
orders.show(3, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- items_count: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



événements : 50000 | restaurants : 200


+-----------+-------------+-----------+------+----------+-----------+-----------+----------+---------+-------------------+
|order_id   |restaurant_id|customer_id|city  |amount_mad|items_count|distance_km|courier_id|status   |event_time         |
+-----------+-------------+-----------+------+----------+-----------+-----------+----------+---------+-------------------+
|ORD-0000001|R-168        |C-11420    |Tanger|192.32    |2          |2.9        |L-TAN-05  |CREATED  |2026-10-16 00:18:01|
|ORD-0000001|R-168        |C-11420    |Tanger|192.32    |2          |2.9        |L-TAN-05  |PICKED_UP|2026-10-16 00:31:34|
|ORD-0000002|R-024        |C-15240    |Tanger|84.46     |2          |2.1        |L-TAN-08  |CREATED  |2026-10-16 00:39:06|
+-----------+-------------+-----------+------+----------+-----------+-----------+----------+---------+-------------------+
only showing top 3 rows



## 2. CA par ville — API DataFrame puis SQL

Même règle qu'en Partie 2 : commandes `DELIVERED`, une seule ligne par `order_id` (suppression des doublons).

In [3]:
delivered = (orders.filter(F.col("status") == "DELIVERED")
                   .dropDuplicates(["order_id"]))

ca_df = (delivered.groupBy("city")
                  .agg(F.round(F.sum("amount_mad"), 2).alias("ca_mad"),
                       F.count("*").alias("commandes_livrees"),
                       F.round(F.avg("amount_mad"), 2).alias("panier_moyen_mad"))
                  .orderBy(F.desc("ca_mad")))
ca_df.show()

+----------+----------+-----------------+----------------+
|      city|    ca_mad|commandes_livrees|panier_moyen_mad|
+----------+----------+-----------------+----------------+
|Casablanca|1079464.82|             6874|          157.04|
|     Rabat| 596828.52|             3342|          178.58|
| Marrakech| 503885.09|             2834|           177.8|
|    Tanger| 410253.37|             2263|          181.29|
+----------+----------+-----------------+----------------+



In [4]:
orders.createOrReplaceTempView("orders")

ca_sql = spark.sql('''
    WITH livrees AS (
        SELECT DISTINCT order_id, city, amount_mad      -- 1 ligne par commande
        FROM orders
        WHERE status = 'DELIVERED'
    )
    SELECT city,
           ROUND(SUM(amount_mad), 2) AS ca_mad,
           COUNT(*)                  AS commandes_livrees,
           ROUND(AVG(amount_mad), 2) AS panier_moyen_mad
    FROM livrees
    GROUP BY city
    ORDER BY ca_mad DESC
''')
ca_sql.show()

# Les deux versions donnent exactement le même résultat
print("Résultats identiques DataFrame / SQL :",
      sorted(ca_df.collect()) == sorted(ca_sql.collect()))

+----------+----------+-----------------+----------------+
|      city|    ca_mad|commandes_livrees|panier_moyen_mad|
+----------+----------+-----------------+----------------+
|Casablanca|1079464.82|             6874|          157.04|
|     Rabat| 596828.52|             3342|          178.58|
| Marrakech| 503885.09|             2834|           177.8|
|    Tanger| 410253.37|             2263|          181.29|
+----------+----------+-----------------+----------------+



Résultats identiques DataFrame / SQL : True


### Mesure des temps (même calcul complet qu'en Partie 2, sans cache)

Lecture MinIO → filtre → dédoublonnage → agrégation, 3 essais pour chaque API.

In [5]:
def ca_dataframe():
    return (spark.read.schema(ORDER_SCHEMA).json(HISTORY)
                 .filter(F.col("status") == "DELIVERED")
                 .dropDuplicates(["order_id"])
                 .groupBy("city").agg(F.sum("amount_mad")).collect())

def ca_sql_query():
    return spark.sql('''SELECT city, SUM(amount_mad) FROM
                          (SELECT DISTINCT order_id, city, amount_mad FROM orders WHERE status = 'DELIVERED')
                        GROUP BY city''').collect()

for i in range(3):
    chrono(f"CA par ville DataFrame sans cache - essai {i + 1}", ca_dataframe)
for i in range(3):
    chrono(f"CA par ville SQL sans cache - essai {i + 1}", ca_sql_query)

[temps] CA par ville DataFrame sans cache - essai 1 : 7.577 s


[temps] CA par ville DataFrame sans cache - essai 2 : 5.464 s


[temps] CA par ville DataFrame sans cache - essai 3 : 3.619 s


[temps] CA par ville SQL sans cache - essai 1 : 3.593 s


[temps] CA par ville SQL sans cache - essai 2 : 3.463 s


[temps] CA par ville SQL sans cache - essai 3 : 2.947 s


## 3. Jointure avec le référentiel restaurants : CA par catégorie et par ville

Les deux tables ont une colonne `city` : on renomme celle du référentiel pour éviter l'ambiguïté.

In [6]:
ref = restaurants.select("restaurant_id",
                         F.col("name").alias("restaurant_name"),
                         F.col("city").alias("restaurant_city"),
                         "category")

joined = delivered.join(ref, on="restaurant_id", how="left")

ca_cat_ville = (joined.groupBy("city", "category")
                      .agg(F.round(F.sum("amount_mad"), 2).alias("ca_mad"),
                           F.count("*").alias("commandes"))
                      .orderBy("city", F.desc("ca_mad")))
ca_cat_ville.show(50)

# Contrôle qualité de la jointure
print("commandes sans restaurant connu :", joined.filter(F.col("category").isNull()).count())

+----------+---------+---------+---------+
|      city| category|   ca_mad|commandes|
+----------+---------+---------+---------+
|Casablanca|    tacos|257056.39|     1952|
|Casablanca|  healthy|150831.54|      766|
|Casablanca|asiatique|137683.86|      749|
|Casablanca|    sushi|126989.22|      540|
|Casablanca|    pizza|120439.94|      777|
|Casablanca|   burger|119160.12|      822|
|Casablanca|   poulet|101301.34|      868|
|Casablanca| marocain| 66002.41|      400|
| Marrakech|    sushi| 157947.9|      653|
| Marrakech| marocain| 84687.52|      490|
| Marrakech|   burger| 69527.24|      473|
| Marrakech|  healthy| 56667.77|      287|
| Marrakech|    tacos| 53652.86|      410|
| Marrakech|    pizza| 49879.45|      316|
| Marrakech|asiatique| 23937.24|      144|
| Marrakech|   poulet|  7585.11|       61|
|     Rabat|  healthy|252472.38|     1243|
|     Rabat|    sushi|106876.81|      463|
|     Rabat|    tacos|  67079.3|      518|
|     Rabat|    pizza| 50189.27|      324|
|     Rabat

commandes sans restaurant connu : 0


In [7]:
# Vue croisée ville x catégorie (CA en MAD) pour le rapport
pivot = (ca_cat_ville.groupBy("category").pivot("city").agg(F.first("ca_mad"))
                     .orderBy("category").toPandas().set_index("category"))
pivot["Total"] = pivot.sum(axis=1)
pivot.sort_values("Total", ascending=False).round(0)

,Casablanca,Marrakech,Rabat,Tanger,Total
category,,,,,
healthy,150832.0,56668.0,252472.0,92527.0,552499.0
sushi,126989.0,157948.0,106877.0,144557.0,536371.0
tacos,257056.0,53653.0,67079.0,29983.0,407772.0
burger,119160.0,69527.0,42408.0,85342.0,316437.0
pizza,120440.0,49879.0,50189.0,NaN,220509.0
asiatique,137684.0,23937.0,13369.0,36185.0,211176.0
marocain,66002.0,84688.0,48051.0,9585.0,208326.0
poulet,101301.0,7585.0,16382.0,12074.0,137343.0


## 4. UDF `distance_band` vs version native `when/otherwise`

- **UDF Python** : Spark ne sait pas ce qu'elle fait (boîte noire pour Catalyst). Chaque ligne est
  sérialisée de la JVM vers un processus Python, traitée, puis renvoyée (`BatchEvalPython` dans le plan).
- **`when/otherwise`** : expression native, optimisée par Catalyst et compilée en bytecode JVM par
  Tungsten (*whole-stage code generation*), sans aller-retour vers Python.

In [8]:
@F.udf(returnType=StringType())
def distance_band_udf(distance_km):
    if distance_km is None:
        return None
    if distance_km < 2:
        return "court"
    if distance_km <= 5:
        return "moyen"
    return "long"

def distance_band_native(col):
    return (F.when(col.isNull(), None)
             .when(col < 2, "court")
             .when(col <= 5, "moyen")
             .otherwise("long"))

avec_udf = orders.withColumn("distance_band", distance_band_udf("distance_km"))
sans_udf = orders.withColumn("distance_band", distance_band_native(F.col("distance_km")))

avec_udf.groupBy("distance_band").count().orderBy("distance_band").show()
sans_udf.groupBy("distance_band").count().orderBy("distance_band").show()

+-------------+-----+
|distance_band|count|
+-------------+-----+
|        court|11882|
|         long| 9691|
|        moyen|28427|
+-------------+-----+



+-------------+-----+
|distance_band|count|
+-------------+-----+
|        court|11882|
|         long| 9691|
|        moyen|28427|
+-------------+-----+



In [9]:
# Même calcul, 3 essais chacun (données en cache pour ne mesurer que la colonne calculée)
orders.cache().count()

def run(df):
    return df.groupBy("distance_band").agg(F.count("*"), F.avg("amount_mad")).collect()

for i in range(3):
    chrono(f"distance_band UDF Python - essai {i + 1}", lambda: run(avec_udf))
for i in range(3):
    chrono(f"distance_band when/otherwise - essai {i + 1}", lambda: run(sans_udf))

[temps] distance_band UDF Python - essai 1 : 4.664 s


[temps] distance_band UDF Python - essai 2 : 4.109 s


[temps] distance_band UDF Python - essai 3 : 3.238 s


[temps] distance_band when/otherwise - essai 1 : 1.842 s


[temps] distance_band when/otherwise - essai 2 : 1.485 s


[temps] distance_band when/otherwise - essai 3 : 0.89 s


In [10]:
# Dans le plan physique : BatchEvalPython (UDF) contre une simple expression CASE WHEN (native)
print("----- avec UDF -----")
avec_udf.select("order_id", "distance_band").explain()
print("----- sans UDF -----")
sans_udf.select("order_id", "distance_band").explain()

----- avec UDF -----
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#0, pythonUDF0#3301 AS distance_band#1112]
   +- BatchEvalPython [distance_band_udf(distance_km#6)#1111], [pythonUDF0#3301]
      +- InMemoryTableScan [order_id#0, distance_km#6]
            +- InMemoryRelation [order_id#0, restaurant_id#1, customer_id#2, city#3, amount_mad#4, items_count#5, distance_km#6, courier_id#7, status#8, event_time#9], StorageLevel(disk, memory, deserialized, 1 replicas)
                  +- FileScan json [order_id#0,restaurant_id#1,customer_id#2,city#3,amount_mad#4,items_count#5,distance_km#6,courier_id#7,status#8,event_time#9] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[s3a://lake/bronze/history/orders_history.jsonl], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,restaurant_id:string,customer_id:string,city:string,amount_mad:double,item...


----- sans UDF -----
== Physical Plan ==
AdaptiveSpar

## 5. Écriture du résultat nettoyé en Parquet, partitionné par `city`

Nettoyage : suppression des doublons (`order_id` + `status`), suppression des lignes sans `order_id`,
ajout de `distance_band`, de la catégorie du restaurant et de la date de l'événement.

In [11]:
clean = (orders
         .dropna(subset=["order_id", "status", "event_time"])
         .dropDuplicates(["order_id", "status"])
         .withColumn("distance_band", distance_band_native(F.col("distance_km")))
         .withColumn("event_date", F.to_date("event_time"))
         .join(F.broadcast(ref.select("restaurant_id", "restaurant_name", "category")),
               on="restaurant_id", how="left"))

print("lignes brutes :", orders.count(), "-> lignes nettoyées :", clean.count())

chrono("écriture Parquet silver/orders_history",
       lambda: clean.write.mode("overwrite").partitionBy("city").parquet(SILVER))

lignes brutes : 50000 -> lignes nettoyées : 49018


[temps] écriture Parquet silver/orders_history : 22.6 s


In [12]:
# Contenu de lake/silver/orders_history/ : un sous-dossier city=... par ville
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.Path(SILVER).getFileSystem(sc._jsc.hadoopConfiguration())
it = fs.listFiles(jvm.org.apache.hadoop.fs.Path(SILVER), True)
while it.hasNext():
    st = it.next()
    print(f"{st.getLen():>9} octets  {st.getPath().toString().replace(SILVER, 'silver/orders_history')}")

        0 octets  silver/orders_history/_SUCCESS
   234743 octets  silver/orders_history/city=Casablanca/part-00000-993aaf6b-9abc-434d-9e6d-8aa93979a6bd.c000.snappy.parquet
   359703 octets  silver/orders_history/city=Casablanca/part-00001-993aaf6b-9abc-434d-9e6d-8aa93979a6bd.c000.snappy.parquet
   102677 octets  silver/orders_history/city=Marrakech/part-00000-993aaf6b-9abc-434d-9e6d-8aa93979a6bd.c000.snappy.parquet
   149903 octets  silver/orders_history/city=Marrakech/part-00001-993aaf6b-9abc-434d-9e6d-8aa93979a6bd.c000.snappy.parquet
   113987 octets  silver/orders_history/city=Rabat/part-00000-993aaf6b-9abc-434d-9e6d-8aa93979a6bd.c000.snappy.parquet
   174676 octets  silver/orders_history/city=Rabat/part-00001-993aaf6b-9abc-434d-9e6d-8aa93979a6bd.c000.snappy.parquet
    78933 octets  silver/orders_history/city=Tanger/part-00000-993aaf6b-9abc-434d-9e6d-8aa93979a6bd.c000.snappy.parquet
   119434 octets  silver/orders_history/city=Tanger/part-00001-993aaf6b-9abc-434d-9e6d-8aa93979a6bd

In [13]:
# Relecture : le filtre sur city ne lit qu'un seul dossier (partition pruning)
silver = spark.read.parquet(SILVER)
casa = silver.filter(F.col("city") == "Casablanca")
print("lignes Casablanca :", casa.count())
casa.explain()          # PartitionFilters: [isnotnull(city), (city = Casablanca)]

lignes Casablanca : 22003


== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [restaurant_id#4449,order_id#4450,customer_id#4451,amount_mad#4452,items_count#4453,distance_km#4454,courier_id#4455,status#4456,event_time#4457,distance_band#4458,event_date#4459,restaurant_name#4460,category#4461,city#4462] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://lake/silver/orders_history], PartitionFilters: [isnotnull(city#4462), (city#4462 = Casablanca)], PushedFilters: [], ReadSchema: struct<restaurant_id:string,order_id:string,customer_id:string,amount_mad:double,items_count:int,...




## 6. `explain(True)` sur la jointure : plans logique, optimisé (Catalyst) et physique

In [14]:
requete_jointure = (orders.filter(F.col("status") == "DELIVERED")
                          .join(ref, "restaurant_id")
                          .groupBy("city", "category")
                          .agg(F.sum("amount_mad").alias("ca_mad")))
requete_jointure.explain(True)

with open(f"{RESULTS}/partie3_explain_jointure.txt", "w") as f:
    f.write(requete_jointure._jdf.queryExecution().toString())

== Parsed Logical Plan ==
'Aggregate ['city, 'category], ['city, 'category, sum('amount_mad) AS ca_mad#4524]
+- Project [restaurant_id#1, order_id#0, customer_id#2, city#3, amount_mad#4, items_count#5, distance_km#6, courier_id#7, status#8, event_time#9, restaurant_name#641, restaurant_city#642, category#23]
   +- Join Inner, (restaurant_id#1 = restaurant_id#20)
      :- Filter (status#8 = DELIVERED)
      :  +- Relation [order_id#0,restaurant_id#1,customer_id#2,city#3,amount_mad#4,items_count#5,distance_km#6,courier_id#7,status#8,event_time#9] json
      +- Project [restaurant_id#20, name#21 AS restaurant_name#641, city#22 AS restaurant_city#642, category#23]
         +- Relation [restaurant_id#20,name#21,city#22,category#23,latitude#24,longitude#25] json

== Analyzed Logical Plan ==
city: string, category: string, ca_mad: double
Aggregate [city#3, category#23], [city#3, category#23, sum(amount_mad#4) AS ca_mad#4524]
+- Project [restaurant_id#1, order_id#0, customer_id#2, city#3, amou

**Lecture du plan :**

1. *Parsed / Analyzed Logical Plan* : la requête telle qu'écrite (`Filter` au-dessus de la lecture, `Join`, `Aggregate`), puis avec les colonnes et types résolus grâce au schéma.
2. *Optimized Logical Plan* : Catalyst a ajouté `isnotnull(status)` et `isnotnull(restaurant_id)` au filtre, l'a placé **sous** la jointure (*predicate pushdown*) et a **supprimé les colonnes inutiles** (*column pruning*) : côté commandes seules `restaurant_id`, `city`, `amount_mad` remontent, côté référentiel seules `restaurant_id` et `category` sont lues (`ReadSchema: struct<restaurant_id,category>`, `PushedFilters: [IsNotNull(restaurant_id)]`). Comme `orders` est en cache, le filtre s'applique sur l'`InMemoryTableScan` au lieu d'être poussé jusqu'au fichier JSON.
3. *Physical Plan* : `BroadcastHashJoin ... BuildRight` — le référentiel (200 lignes, quelques Ko) est sous le seuil `spark.sql.autoBroadcastJoinThreshold` (10 Mo) : il est envoyé à chaque Executor (`BroadcastExchange`) et la grosse table n'est **pas** shufflée pour la jointure. Le seul `Exchange hashpartitioning(city, category)` sert à l'agrégation (après un `partial_sum` local).


In [15]:
# Pour comparaison : si on interdit le broadcast, Spark choisit un SortMergeJoin (2 shuffles + tris)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "false")      # sinon AQE peut repasser en broadcast
smj = (orders.filter(F.col("status") == "DELIVERED").join(ref, "restaurant_id")
             .groupBy("city", "category").agg(F.sum("amount_mad")))
smj.explain()
_ = chrono("jointure SortMergeJoin (broadcast désactivé)", smj.collect)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
spark.conf.set("spark.sql.adaptive.enabled", "true")
_ = chrono("jointure BroadcastHashJoin", requete_jointure.collect)

== Physical Plan ==
*(6) HashAggregate(keys=[city#3, category#23], functions=[sum(amount_mad#4)])
+- Exchange hashpartitioning(city#3, category#23, 8), ENSURE_REQUIREMENTS, [plan_id=4172]
   +- *(5) HashAggregate(keys=[city#3, category#23], functions=[partial_sum(amount_mad#4)])
      +- *(5) Project [city#3, amount_mad#4, category#23]
         +- *(5) SortMergeJoin [restaurant_id#1], [restaurant_id#20], Inner
            :- *(2) Sort [restaurant_id#1 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(restaurant_id#1, 8), ENSURE_REQUIREMENTS, [plan_id=4155]
            :     +- *(1) Project [restaurant_id#1, city#3, amount_mad#4]
            :        +- *(1) Filter ((isnotnull(status#8) AND (status#8 = DELIVERED)) AND isnotnull(restaurant_id#1))
            :           +- InMemoryTableScan [amount_mad#4, city#3, restaurant_id#1, status#8], [isnotnull(status#8), (status#8 = DELIVERED), isnotnull(restaurant_id#1)]
            :                 +- InMemoryRelation [ord

[temps] jointure SortMergeJoin (broadcast désactivé) : 4.8 s


[temps] jointure BroadcastHashJoin : 2.833 s


## 7. Sauvegarde des résultats et comparaison avec la Partie 2

In [16]:
ca_df.toPandas().to_csv(f"{RESULTS}/partie3_ca_par_ville.csv", index=False)
ca_cat_ville.toPandas().to_csv(f"{RESULTS}/partie3_ca_categorie_ville.csv", index=False)
pivot.to_csv(f"{RESULTS}/partie3_pivot_categorie_ville.csv")
with open(f"{RESULTS}/partie3_timings.json", "w") as f:
    json.dump(timings, f, indent=2, ensure_ascii=False)

# Comparaison RDD (Partie 2) / DataFrame / SQL sur le même calcul complet
p2 = json.load(open(f"{RESULTS}/partie2_timings.json"))
def moyenne(d, prefixe):
    v = [t for k, t in d.items() if k.startswith(prefixe)]
    return round(sum(v) / len(v), 3), round(min(v), 3)

comparaison = pd.DataFrame(
    [("RDD (Partie 2)", *moyenne(p2, "CA par ville RDD sans cache")),
     ("DataFrame", *moyenne(timings, "CA par ville DataFrame sans cache")),
     ("Spark SQL", *moyenne(timings, "CA par ville SQL sans cache"))],
    columns=["API", "temps moyen (s)", "meilleur temps (s)"])
comparaison.to_csv(f"{RESULTS}/partie3_comparaison_rdd_df_sql.csv", index=False)
comparaison

,API,temps moyen (s),meilleur temps (s)
0,RDD (Partie 2),7.404,5.900
1,DataFrame,5.553,3.619
2,Spark SQL,3.334,2.947


Ouvrir http://localhost:4040 → onglet **SQL / DataFrame** pour la capture du DAG de la jointure
(on y voit le `BroadcastExchange` et le `BroadcastHashJoin`), puis :

In [17]:
spark.stop()